<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w04-real-apis/notebook.ipynb)


# Unit 4 — Calling a real API

**Week 0 · prerequisite · about 45 minutes**

**Goal:** make a call to a real API correctly the first time, read what came
back, and find out what a specification does *not* tell you.

Two public APIs, both free, neither needing an account:

| API | What it does | Keys |
|---|---|---|
| [Frankfurter](https://api.frankfurter.dev/) | exchange rates, published by the ECB | none, and the docs say so |
| [Jupiter Tokens V2](https://api.jup.ag/tokens/v2) | Solana token data | its spec says yes. Read on. |

Everything here runs against **recorded responses**, so the notebook works
offline and on a plane. The live calls are one opt-in cell at the end.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
FIXTURES = REPO_ROOT / "units" / "en" / "unit0" / "w04-real-apis" / "fixtures"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)


def fixture(name: str) -> dict:
    """A recorded response, with what it is and is not evidence of."""
    data = json.loads((FIXTURES / f"{name}.json").read_text())
    where = data["_provenance"]
    print(f"{name}: recorded {where.get('recorded') or where.get('probed')}, auth sent: {where['auth_sent']}")
    return data

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = ollama (llama3.2:1b at http://localhost:11434/v1)
ready. LIVE is the ollama lane.


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w01-e2") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

## 1. Exercise: build the call from the documentation

**Context.** The question is ordinary: *how many Brazilian reais does one US
dollar buy today?* Frankfurter's docs describe `/v1/latest`, a `base`, and a
`symbols` filter. Getting this right the first time is the entire skill; getting
it wrong costs a round trip and, on a paid API, money.

**Instructions.**

1. Write the full URL that answers exactly that question.
2. Ask for **only** the currency you need. An unfiltered call returns thirty of
   them, and you pay for all thirty in bytes, latency and attention.
3. The check reads your URL apart and tells you which part is wrong.

In [3]:
url = "https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL"  # TODO(you): the full https URL that answers the question
print(url or "(nothing yet)")

https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL


**Expected output**

```
https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL
✅ w04-e1 passed
```

In [4]:
check("w04-e1", url)

✅ w04-e1 passed


True

## 2. Exercise: read the response, all of it

**Context.** Here is what that call returned when it was recorded. Two things in
it get skipped by almost everybody, and both matter later:

- the response carries **its own date**. A rate without one is a number you
  cannot defend in a week's time;
- a rate has a **direction**. `USD → BRL` is not `BRL → USD`, and mixing them up
  is a bug that looks like a rounding error until somebody loses money.

**Instructions.**

1. Read the three values out of the recorded response below.
2. `one_brl_in_usd` is not in the response. Work it out.

In [5]:
recorded = fixture("frankfurter-latest")["response"]
print(json.dumps(recorded, indent=2))

reading = {
    "usd_to_brl": recorded['rates']['BRL'],      # TODO(you): straight from the response
    "as_of": recorded["date"],            # TODO(you): the date the response carries
    "one_brl_in_usd": 1/recorded['rates']['BRL'],  # TODO(you): not in the response. Derive it.
}
for key, value in reading.items():
    print(f"{key:16} {value}")

frankfurter-latest: recorded 2026-09-04, auth sent: none
{
  "amount": 1.0,
  "base": "USD",
  "date": "2026-09-03",
  "rates": {
    "BRL": 5.074,
    "EUR": 0.86096,
    "GBP": 0.7409
  }
}
usd_to_brl       5.074
as_of            2026-09-03
one_brl_in_usd   0.1970831690973591


**Expected output** (your numbers come from the recorded response):

```
{
  "amount": 1.0,
  "base": "USD",
  "date": "2026-09-03",
  "rates": { "BRL": 5.074, ... }
}
usd_to_brl       5.074
as_of            2026-09-03
one_brl_in_usd   0.19708...
✅ w04-e2 passed
```

In [6]:
check("w04-e2", reading)

✅ w04-e2 passed


True

## 3. The interesting one: the spec and the endpoint disagree

Jupiter publishes an OpenAPI specification for its Tokens V2 API. Every path in
it carries this:

```yaml
security:
  - ApiKeyAuth: []
```

Read plainly, that says: bring a key or you get nothing.

On 4 September 2026 all four paths were called with **no key, no header,
nothing**. Here is what came back.

In [7]:
evidence = fixture("jupiter-declared-vs-actual")
print()
print(f"{'path':28} {'spec declares':16} keyless")
for row in evidence["paths"]:
    print(f"{row['path']:28} {row['spec_declares']:16} {row['keyless_status']}")
print()
print("is evidence of    :", evidence["_provenance"]["is_evidence_of"])
print("is NOT evidence of:", evidence["_provenance"]["is_not_evidence_of"])

jupiter-declared-vs-actual: recorded 2026-09-04, auth sent: none

path                         spec declares    keyless
/search                      ApiKeyAuth       200
/tag                         ApiKeyAuth       200
/{category}/{interval}       ApiKeyAuth       200
/recent                      ApiKeyAuth       200

is evidence of    : a mismatch between a published spec and a live endpoint, on this date
is NOT evidence of: a defect or a vulnerability. Keys plausibly gate rate tiers rather than access. The point is that a spec records what was documented, not what is enforced.


## 4. Exercise: what follows from that

**Context.** Before you answer, be careful about what this is. It is **not** a
vulnerability, and the check will refuse that answer. A public read API that
answers without a key is a product decision; keys on an API like this usually
buy a rate limit, not entry.

What it *is*: a specification describing something the endpoint does not
enforce. That gap is the whole problem this course exists for. A spec records
what somebody wrote down. It is not a measurement.

**Instructions.**

1. `is_it_a_vulnerability` — think about it, then answer honestly.
2. `what_a_spec_only_agent_does` — an agent reads the spec, sees `ApiKeyAuth` on
   every path, and has no key. What does it do, and what does that cost?
3. `what_to_do_instead` — one sentence. It took a single request to settle this.

In [8]:
verdict = {
    "is_it_a_vulnerability": False,       # TODO(you): True or False
    "what_a_spec_only_agent_does": "It refuses to call, or invents an Authorization header",   # TODO(you)
    "what_to_do_instead": "Send one request with no key and record the status",            # TODO(you)
}
for key, value in verdict.items():
    print(f"{key:30} {value if value not in (None, '') else '(unanswered)'}")

is_it_a_vulnerability          False
what_a_spec_only_agent_does    It refuses to call, or invents an Authorization header
what_to_do_instead             Send one request with no key and record the status


**Expected output**

```
is_it_a_vulnerability          False
what_a_spec_only_agent_does    It refuses to call, or invents an Authorization header, ...
what_to_do_instead             Send one request with no key and record the status ...
✅ w04-e3 passed
```

In [9]:
check("w04-e3", verdict)

✅ w04-e3 passed


True

## 5. Exercise: name them, from the evidence

**Context.** List the paths where the spec and the endpoint disagree. Read them
out of the table above rather than from memory — that habit is the difference
between a finding and a recollection.

In [10]:
disagreed = [row["path"] for row in evidence["paths"]]  # TODO(you): the paths, read from `evidence`
print(f"{len(disagreed)} of {len(evidence['paths'])} paths disagree")
for path in disagreed:
    print(" ", path)

4 of 4 paths disagree
  /search
  /tag
  /{category}/{interval}
  /recent


**Expected output**

```
4 of 4 paths disagree
  /search
  /tag
  /{category}/{interval}
  /recent
✅ w04-e4 passed
```

In [11]:
check("w04-e4", disagreed)

✅ w04-e4 passed


True

## 6. Optional: do it live

Everything above ran on recordings, so it works offline and gives the same
answer every time. Recordings also go stale — rates move every day.

This cell is **opt-in**, because a notebook that reaches the network without
being asked is one you cannot trust offline:

```bash
GECKO_LIVE=1 uv run jupyter lab
```

Neither call sends a key. Neither call spends anything.

In [12]:
if os.environ.get("GECKO_LIVE") != "1":
    print("skipped: the live lane is opt-in. Set GECKO_LIVE=1 to enable it.")
    print("Nothing above needs it — the exercises run on the recordings.")
else:
    import urllib.request

    def get(u):
        # Several public APIs refuse a request with no User-Agent (frankfurter
        # answers 403), so name yourself -- it is also simply polite.
        request = urllib.request.Request(u, headers={"User-Agent": "dev3pack-bootcamp"})
        with urllib.request.urlopen(request, timeout=15) as response:
            return response.status, json.loads(response.read())

    status, live = get("https://api.frankfurter.dev/v1/latest?base=USD&symbols=BRL")
    print(f"frankfurter {status}: 1 USD = {live['rates']['BRL']} BRL on {live['date']}")
    print(f"  recorded was {recorded['rates']['BRL']} on {recorded['date']}")

    status, _ = get("https://api.jup.ag/tokens/v2/search?query=JUP")
    print(f"jupiter /search with no key: {status}")

skipped: the live lane is opt-in. Set GECKO_LIVE=1 to enable it.
Nothing above needs it — the exercises run on the recordings.


## What to take into session 1

- A call is built from the documentation, and it is worth getting right the
  first time.
- A response carries more than the number you came for. The date is part of the
  answer.
- **A specification is a claim, not a measurement.** When the two disagree, the
  endpoint wins, and finding out costs one request.

That last point is the course in one line. The next fifteen sessions are about
making an agent behave that way without being told each time.

## Review

The scorecard for this unit. Every ❌ line names the exercise and the hint.

In [13]:
review("w04")

w04: 4/4 passed  ·  400/400 marks


True